# 02. NetHealth 기준선 비교 (모듈 1)

1. **개인 vs 집단 기준선** — 라벨 없이, 두 방법이 같은 날·같은 사람에게 울리는지 비교 (`ev.compare_alarms`)
2. **가짜 이탈 주입 평가** — 정상 구간 14일에 일부러 변화를 넣고 잡아내는지 (`ev.evaluate_injection`)

공통 약속
- 경보 기준: 오경보 사람당 30일 1회 = **3.3%** (`ev.ALARM_RATE`). 방법마다 기준값 **하나**를 모든 사람에게 똑같이
- 방향 통일: '클수록 나쁨' (걸음↓, 수면↓, 잠든 시각↑, 심박↑) — `bl.bad_score`
- 개인 기준선 = 직전 **달력** 28일, z는 ±8로 자름. 집단 기준선 = 같은 날 30명 이상
- 결과는 **사람마다** 계산해서 분포(중앙값, 박스플롯, Wilcoxon)로 봄

결과를 남길 때만 `WHO`를 내 이름으로, `SAVE = True`로 바꿔 실행 (기본은 `results/results.csv`에 기록 안 함)

In [ ]:
import sys, warnings
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import font_manager

# 한글 글꼴 (윈도우: 맑은 고딕, 맥: AppleGothic, 그 외: 나눔고딕)
for f in ['Malgun Gothic', 'AppleGothic', 'NanumGothic']:
    if f in {x.name for x in font_manager.fontManager.ttflist}:
        plt.rcParams['font.family'] = f
        break
plt.rcParams['axes.unicode_minus'] = False
pd.set_option('display.width', 160)

from src import data_loader as dl, preprocessing as pp, baseline as bl, evaluation as ev

In [ ]:
with warnings.catch_warnings():
    warnings.simplefilter('ignore')          # Calendar.csv가 없을 때 나오는 경고 숨김
    nh = dl.load_nethealth()
print('NetHealth:', nh.shape, '| 사람 수:', nh['pid'].nunique())
WHO = '이름'          # log_result에 남길 이름
SAVE = False         # True일 때만 results.csv에 기록

---
## 1. 개인 vs 집단 기준선 (라벨 없음)
- **같은 날 비율**: 개인 기준선이 울린 날 중 집단 기준선도 울린 비율. 낮을수록 두 방법이 '다른 날'을 잡음
- **0번인 사람**: 경보가 한 번도 안 울린 사람 수
- **상위 10% 몫**: 경보를 가장 많이 받은 10% 사람이 전체 경보에서 차지하는 비율 (높을수록 몇 사람에게 몰림)
- **사람별 오경보율**: 라벨이 없으니 모든 경보를 오경보로 볼 때 사람별 경보율 (중앙값 [1사분위, 3사분위])

In [ ]:
def compare_table(dow_adjust=False):
    rows = []
    for c in ['steps', 'sleep_min', 'bed_min']:
        d = nh.assign(p=bl.bad_score(bl.personal_z(nh, c, dow_adjust=dow_adjust), c),
                      g=bl.bad_score(bl.group_z(nh, c, min_n=30), c))
        r = ev.compare_alarms(d, 'p', 'g')
        fa = [f'{m:.3f} [{q1:.3f}, {q3:.3f}]' for m, q1, q3 in r['사람별오경보율']]
        rows.append({'지표': c, '같은 날 비율': round(r['같은날비율'], 3),
                     '0번인 사람 (개인/집단)': r['0번인사람'],
                     '상위 10% 몫 (개인/집단)': tuple(round(v, 3) for v in r['상위10%몫']),
                     '사람별 오경보율 개인': fa[0], '사람별 오경보율 집단': fa[1],
                     '사람 수': r['사람수'], '_raw': r})
    return pd.DataFrame(rows)

cmp = compare_table(dow_adjust=False)
cmp.drop(columns='_raw')

요일 보정을 켜면 (요일 평균은 기준선 창 안의 과거 값으로만 계산):

In [ ]:
cmp_dow = compare_table(dow_adjust=True)
cmp_dow.drop(columns='_raw')

In [ ]:
if SAVE:
    for tag, t in [('요일보정 끔', cmp), ('요일보정 켬', cmp_dow)]:
        for _, r in t.iterrows():
            ev.log_result(WHO, 'NetHealth', '개인 vs 집단 (compare_alarms)',
                          f"{r['지표']}, window=28, group min_n=30, rate=1/30, {tag}", r['_raw'])
    print('results/results.csv에 기록했습니다')
else:
    print('SAVE=False라 기록하지 않음')

---
## 2. 가짜 이탈 주입 평가

사람마다 '정상 구간'(직전 28일에 기록 14일 이상, 구간 안에 기록 7일 이상) 하나를 무작위로 골라 **14일** 동안 지표를 바꿈.
- **탐지**: 주입 구간 14일 안에 하루라도 경보 → 탐지율 (주 지표)
- **대조군**: 같은 구간을 주입 없이 돌렸을 때 울리면 오경보 → 오경보율
- **차이** = 탐지율 − 오경보율 (Youden's J). 0이면 우연 수준
- **탐지 지연**: 주입 시작 후 첫 경보까지 며칠 (0 = 첫날)
- 기준값: 대조군 전체에서 하루 오경보율 3.3%가 되도록 방법마다 하나. CUSUM은 경보선 h를 그렇게 맞춤
- 반복 10번(시드 0~9)으로 사람마다 비율을 낸 뒤, 표의 숫자는 **사람별 값의 중앙값**
- PR-AUC는 보조 (주입 구간 날 vs 같은 구간 대조군 날)

비교 방법: 무작위 / 단일 z(|z|) / 개인 기준선 / 개인 기준선 + CUSUM / 집단 기준선

In [ ]:
SCENARIOS = [
    # (지표, 크기, 'relative'=비율 | 'absolute'=더하기, 모양)
    ('steps',   -0.3, 'relative', 'sudden'),    # 걸음 30% 감소, 갑자기
    ('steps',   -0.3, 'relative', 'gradual'),   # 걸음 30% 감소, 서서히
    ('bed_min', 120,  'absolute', 'sudden'),    # 잠든 시각 2시간 늦어짐, 갑자기
    ('bed_min', 120,  'absolute', 'gradual'),   # 잠든 시각 2시간 늦어짐, 서서히
]
N_REPS, SEED = 10, 0

results = {}
for col, eff, mode, shape in SCENARIOS:
    key = f'{col} {eff:+g} {shape}'
    results[key] = ev.evaluate_injection(nh, col, eff, mode=mode, shape=shape, n_reps=N_REPS, seed=SEED)
    print('완료:', key, '| CUSUM h =', round(results[key]['setting']['cusum_h'], 2))

In [ ]:
for key, res in results.items():
    print(f'■ {key}')
    display(ev.results_table(res))

### 사람별 분포 (박스플롯)

In [ ]:
fig, axes = plt.subplots(len(results), 2, figsize=(14, 3.6 * len(results)))
for i, res in enumerate(results.values()):
    ev.plot_person_box(res, '차이', ax=axes[i, 0])
    ev.plot_person_box(res, '탐지지연', ax=axes[i, 1])
plt.tight_layout(); plt.show()

### 방법 간 Wilcoxon 검정 (사람 단위로 짝지어 '차이' 비교, 기준 = 개인 기준선)

In [ ]:
for key, res in results.items():
    print(f'■ {key}')
    display(ev.wilcoxon_table(res, metric='차이', ref='개인 기준선'))

### 요일 보정 켜고 한 번 더 (걸음 −30%, 갑자기)

In [ ]:
res_dow = ev.evaluate_injection(nh, 'steps', -0.3, shape='sudden', n_reps=N_REPS, seed=SEED, dow_adjust=True)
ev.results_table(res_dow)

### 결과 기록

In [ ]:
if SAVE:
    for key, res in list(results.items()) + [('steps -0.3 sudden 요일보정', res_dow)]:
        s = res['setting']
        ev.log_table(ev.results_table(res), WHO, 'NetHealth',
                     f"주입 {key}, length={s['length']}, reps={s['n_reps']}, seed={s['seed']}, "
                     f"rate=1/30, dow_adjust={s['dow_adjust']}, k={s['k']}")
    display(pd.read_csv(dl.PATHS['results'], encoding='utf-8-sig').tail(5))
else:
    print('SAVE=False라 기록하지 않음')